# Ders 4: Spektral Analiz
- 4.1 Spektral Yoğunluklar
- 4.2 Periyodogram
- 4.3 Zaman-Değişmez Doğrusal Filtreler
- 4.4 ARMA Sürecinin Spektral Yoğunluğu


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
from scipy.fft import fft, fftfreq
import warnings; warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 4)
print("Hazır!")


## 4.1 Spektral Yoğunluklar

Durağan bir sürecin **spektral yoğunluk fonksiyonu** (power spectral density):
$$f(\lambda) = \frac{1}{2\pi} \sum_{h=-\infty}^{\infty} \gamma(h) e^{-ih\lambda}, \quad \lambda \in [-\pi, \pi]$$

**Ters dönüşüm (Herglotz teoremi):**
$$\gamma(h) = \int_{-\pi}^{\pi} e^{ih\lambda} f(\lambda) d\lambda$$

Özel olarak: $\gamma(0) = \text{Var}(X_t) = \int_{-\pi}^{\pi} f(\lambda) d\lambda$

Yorum: $f(\lambda)\Delta\lambda$ ≈ $[\lambda, \lambda+\Delta\lambda]$ frekans bandındaki varyans katkısı


In [ ]:
# ── Teorik Spektral Yoğunluk Hesaplama ──

def theoretical_spectral_density(ar_params, ma_params, sigma2=1.0, n_freq=500):
    '''
    ARMA(p,q) için teorik spektral yoğunluk:
    f(λ) = (σ²/2π) |θ(e^{iλ})|² / |φ(e^{iλ})|²
    '''
    lambdas = np.linspace(-np.pi, np.pi, n_freq)
    f = np.zeros(n_freq)
    
    for k, lam in enumerate(lambdas):
        z = np.exp(1j * lam)
        
        # AR polinomu: φ(z) = 1 - φ₁z - φ₂z² - ...
        ar_poly = 1.0
        for j, phi in enumerate(ar_params):
            ar_poly -= phi * z**(j+1)
        
        # MA polinomu: θ(z) = 1 + θ₁z + θ₂z² + ...
        ma_poly = 1.0
        for j, theta in enumerate(ma_params):
            ma_poly += theta * z**(j+1)
        
        f[k] = (sigma2 / (2 * np.pi)) * np.abs(ma_poly)**2 / np.abs(ar_poly)**2
    
    return lambdas, f

# Farklı modeller
models = {
    'Beyaz Gürültü': ([], [], 'gray'),
    'AR(1) φ=0.9': ([0.9], [], 'steelblue'),
    'AR(1) φ=-0.9': ([-0.9], [], 'crimson'),
    'AR(2) kompleks': ([1.0, -0.5], [], 'green'),
    'MA(1) θ=0.8': ([], [0.8], 'orange'),
    'ARMA(1,1)': ([0.7], [0.4], 'purple'),
}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for idx, (name, (ar, ma, color)) in enumerate(models.items()):
    lam, f = theoretical_spectral_density(ar, ma)
    
    axes[idx].plot(lam, f, color=color, lw=2)
    axes[idx].fill_between(lam, 0, f, alpha=0.15, color=color)
    axes[idx].set_title(f'{name}', fontsize=11)
    axes[idx].set_xlabel('Frekans λ')
    axes[idx].set_ylabel('f(λ)')
    axes[idx].set_xlim(-np.pi, np.pi)
    axes[idx].set_xticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
    axes[idx].set_xticklabels(['-π', '-π/2', '0', 'π/2', 'π'])
    
    # Toplam varyans kontrolü
    var_approx = np.trapezoid(f, lam)
    axes[idx].set_xlabel(f'Frekans λ  [∫f dλ = {var_approx:.3f} ≈ γ(0)]')

plt.tight_layout()
plt.show()
print("Yorumlar:")
print("  AR φ>0: Düşük frekanslarda yüksek güç (yavaş dalgalanma)")
print("  AR φ<0: Yüksek frekanslarda yüksek güç (hızlı dalgalanma)")
print("  Beyaz gürültü: Tüm frekanslarda eşit güç (düz spektrum)")


## 4.2 Periyodogram

Örneklem spektrumu için **periyodogram**:
$$I(\lambda_j) = \frac{1}{n}\left|\sum_{t=1}^{n} X_t e^{-it\lambda_j}\right|^2, \quad \lambda_j = \frac{2\pi j}{n}$$

$E[I(\lambda_j)] \approx f(\lambda_j)$ ancak yüksek varyans nedeniyle düzeltme gerekir.

Periyodogram değerleri **asimptotik olarak bağımsız** $\text{Exp}(f(\lambda_j))$ dağılımlıdır.


In [ ]:
# ── Periyodogram Hesaplama ve Yorumlama ──

def periodogram(x):
    n = len(x)
    # FFT tabanlı hesaplama
    Xf = fft(x - x.mean())
    I = (np.abs(Xf)**2) / n
    freqs = fftfreq(n) * 2 * np.pi  # [0, 2π] → [-π, π]
    # Sadece pozitif frekanslar
    pos_idx = freqs > 0
    return freqs[pos_idx], I[pos_idx]

np.random.seed(42)
n = 256

# 3 farklı süreç
processes = {
    'Sinyal (f=0.1) + Gürültü': np.cos(2*np.pi*0.1*np.arange(n)) + np.random.normal(0,0.5,n),
    'AR(1) φ=0.8': None,
    'Beyaz Gürültü': np.random.normal(0, 1, n),
}

from statsmodels.tsa.arima_process import arma_generate_sample
processes['AR(1) φ=0.8'] = arma_generate_sample([1,-0.8],[1], n)

fig, axes = plt.subplots(len(processes), 2, figsize=(14, 10))

for i, (name, X) in enumerate(processes.items()):
    # Sol: Zaman serisi
    axes[i, 0].plot(X, lw=0.8, color='steelblue')
    axes[i, 0].set_title(f'{name} — Zaman Serisi')
    
    # Sağ: Periyodogram
    freqs, I = periodogram(X)
    axes[i, 1].plot(freqs/np.pi, I, lw=0.7, color='steelblue', alpha=0.7)
    axes[i, 1].set_title(f'{name} — Periyodogram')
    axes[i, 1].set_xlabel('Frekans (λ/π)')
    axes[i, 1].set_ylabel('I(λ)')
    
    # Tepe nokta
    peak_idx = np.argmax(I)
    axes[i, 1].axvline(freqs[peak_idx]/np.pi, color='red', ls='--', lw=1.5,
                        label=f'Tepe: λ={freqs[peak_idx]/np.pi:.3f}π')
    axes[i, 1].legend(fontsize=9)

plt.tight_layout()
plt.show()

# Sinüs sinyalini tespit
X_signal = np.cos(2*np.pi*0.1*np.arange(n)) + np.random.normal(0, 0.5, n)
freqs, I = periodogram(X_signal)
peak_freq = freqs[np.argmax(I)] / (2 * np.pi)
print(f"Tespit edilen dominant frekans: {peak_freq:.4f} Hz")
print(f"Beklenen:                        0.1000 Hz")
print(f"Periyot: {1/peak_freq:.1f} örnek")


In [ ]:
# ── Düzgünleştirilmiş Periyodogram (Spectral Estimation) ──
from scipy.signal import welch, periodogram as sp_periodogram

np.random.seed(10)
n = 512
X = arma_generate_sample([1, -0.8], [1], n)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1) Ham periyodogram
freqs_raw, I_raw = sp_periodogram(X, scaling='density')
axes[0].semilogy(freqs_raw * np.pi * 2, I_raw, lw=0.5, color='steelblue')
axes[0].set_title('Ham Periyodogram (yüksek varyans)')
axes[0].set_xlabel('Frekans'); axes[0].set_ylabel('PSD')

# 2) Welch yöntemi (örtüşen pencereler)
for nfft_win in [32, 64, 128]:
    freqs_w, Pxx_w = welch(X, nperseg=nfft_win, scaling='density')
    axes[1].semilogy(freqs_w * np.pi * 2, Pxx_w, lw=1.5, 
                      label=f'Pencere={nfft_win}', alpha=0.8)
axes[1].set_title('Welch Yöntemi (düzgünleştirilmiş)')
axes[1].set_xlabel('Frekans'); axes[1].legend(fontsize=9)

# 3) Teorik vs Welch
lam_th, f_th = theoretical_spectral_density([0.8], [])
freqs_w, Pxx_w = welch(X, nperseg=64, scaling='density')
axes[2].plot(lam_th[lam_th>=0], f_th[lam_th>=0], 'r-', lw=2, label='Teorik f(λ)')
axes[2].semilogy(freqs_w * np.pi * 2, Pxx_w, 'b-', lw=1.5, label='Welch tahmini')
axes[2].set_title('Teorik vs Welch Tahmin'); axes[2].legend()
axes[2].set_xlabel('Frekans [0, π]')

plt.tight_layout(); plt.show()


## 4.3 Zaman-Değişmez Doğrusal Filtreler

**Doğrusal filtre:** $Y_t = \sum_{j=-\infty}^{\infty} a_j X_{t-j}$

**Transfer fonksiyonu:** $A(\lambda) = \sum_j a_j e^{-ij\lambda}$

Filtrelenmiş sürecin spektral yoğunluğu:
$$f_Y(\lambda) = |A(\lambda)|^2 f_X(\lambda)$$

Örnekler:
- **Alçak geçiren (Low-pass) filtre:** Yüksek frekansları keser
- **Yüksek geçiren (High-pass) filtre:** Düşük frekansları keser


In [ ]:
# ── Doğrusal Filtreler: Alçak / Yüksek Geçiren ──
from scipy.signal import butter, filtfilt

np.random.seed(5)
n = 300
t = np.arange(n)

# Karma sinyal: yavaş trend + hızlı salınım + gürültü
slow = 2 * np.sin(2 * np.pi * t / 50)     # Düşük frekans (trend)
fast = 0.5 * np.sin(2 * np.pi * t / 5)    # Yüksek frekans (gürültü benzeri)
noise = np.random.normal(0, 0.3, n)
X = slow + fast + noise

# Butterworth filtreler
def butter_lowpass(cutoff, fs=1.0, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, cutoff / nyq, btype='low')
    return b, a

def butter_highpass(cutoff, fs=1.0, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, cutoff / nyq, btype='high')
    return b, a

b_low, a_low = butter_lowpass(0.1)
b_high, a_high = butter_highpass(0.1)

X_low  = filtfilt(b_low, a_low, X)
X_high = filtfilt(b_high, a_high, X)

fig, axes = plt.subplots(4, 1, figsize=(13, 12), sharex=True)
axes[0].plot(X, 'k', lw=0.8, label='Orijinal X(t)')
axes[0].set_title('Orijinal Seri = Yavaş + Hızlı + Gürültü')
axes[1].plot(slow, 'r--', lw=1.5, label='Gerçek yavaş bileşen'); axes[1].plot(X_low, 'r', lw=1.5, label='LP filtre çıkışı')
axes[1].set_title('Alçak Geçiren Filtre (< 0.1 Hz)'); axes[1].legend()
axes[2].plot(fast+noise, 'b--', lw=1, label='Gerçek hızlı+gürültü'); axes[2].plot(X_high, 'b', lw=1.5, label='HP filtre çıkışı')
axes[2].set_title('Yüksek Geçiren Filtre (> 0.1 Hz)'); axes[2].legend()

# Spektral karşılaştırma
freqs, I_orig  = periodogram(X)
_, I_low   = periodogram(X_low)
_, I_high  = periodogram(X_high)
axes[3].plot(freqs/np.pi, I_orig, 'k', lw=0.7, label='Orijinal', alpha=0.6)
axes[3].plot(freqs/np.pi, I_low, 'r', lw=1.5, label='LP')
axes[3].plot(freqs/np.pi, I_high, 'b', lw=1.5, label='HP')
axes[3].axvline(0.2, color='gray', ls='--', label='Kesme frekansı')
axes[3].set_xlabel('Frekans (λ/π)'); axes[3].set_ylabel('Güç'); axes[3].legend()
axes[3].set_title('Spektral Analiz: Filtre Etkisi')
plt.tight_layout(); plt.show()


## 4.4 ARMA Sürecinin Spektral Yoğunluğu

**ARMA(p,q) için:**
$$f(\lambda) = \frac{\sigma^2}{2\pi} \frac{|\theta(e^{i\lambda})|^2}{|\phi(e^{i\lambda})|^2}$$

**Özel durumlar:**
- AR(1): $f(\lambda) = \frac{\sigma^2}{2\pi(1 + \phi^2 - 2\phi\cos\lambda)}$
- MA(1): $f(\lambda) = \frac{\sigma^2}{2\pi}(1 + \theta^2 + 2\theta\cos\lambda)$


In [ ]:
# ── ARMA Spektral Yoğunluğu: Teori vs Periyodogram ──
np.random.seed(99)
n = 1024  # Büyük n → daha iyi tahmin

model_params = [
    ('AR(1) φ=0.8',  [0.8],    []),
    ('AR(2)',         [1.0,-0.5],[]),
    ('MA(1) θ=0.7',  [],       [0.7]),
    ('ARMA(1,1)',     [0.7],    [0.5]),
]

fig, axes = plt.subplots(1, len(model_params), figsize=(16, 4))

for idx, (name, ar, ma) in enumerate(model_params):
    # Simüle et
    ar_poly = np.array([1] + [-p for p in ar])
    ma_poly = np.array([1] + list(ma))
    X = arma_generate_sample(ar_poly, ma_poly, n, scale=1)
    
    # Periyodogram (düzgünleştirilmiş)
    freqs_w, Pxx = welch(X, nperseg=128, scaling='density')
    
    # Teorik spektrum
    lam_th, f_th = theoretical_spectral_density(ar, ma, sigma2=1.0)
    
    axes[idx].plot(lam_th[lam_th >= 0], f_th[lam_th >= 0], 
                    'r-', lw=2.5, label='Teorik f(λ)', alpha=0.8)
    axes[idx].plot(freqs_w * 2 * np.pi, Pxx / (2*np.pi), 
                    'b-', lw=1, label='Welch tahmini', alpha=0.7)
    axes[idx].set_title(name, fontsize=10)
    axes[idx].set_xlabel('Frekans λ ∈ [0, π]')
    axes[idx].legend(fontsize=8)
    axes[idx].set_xlim(0, np.pi)

axes[0].set_ylabel('Spektral Yoğunluk f(λ)')
plt.tight_layout(); plt.show()

# Rasyonel spektral yoğunluk tahmini için
print("Spectral peak analizi:")
for name, ar, ma in model_params:
    lam_th, f_th = theoretical_spectral_density(ar, ma)
    peak_lam = lam_th[np.argmax(f_th)]
    print(f"  {name:<20}: Peak λ = {peak_lam:.4f} rad ({peak_lam/np.pi:.3f}π)")


## Özet: Spektral Analiz

| Kavram | Tanım | Kullanım |
|--------|-------|---------|
| **Spektral Yoğunluk** | $f(\lambda) = \frac{1}{2\pi}\sum \gamma(h)e^{-ih\lambda}$ | Teorik güç dağılımı |
| **Periyodogram** | $I(\lambda_j) = \frac{1}{n}\|DFT(X)\|^2$ | Ham spektrum tahmini |
| **Welch Yöntemi** | Örtüşen pencere ortalaması | Düzgünleştirilmiş tahmin |
| **LP/HP Filtre** | $f_Y = |A(\lambda)|^2 f_X$ | Bileşen ayrıştırma |
| **ARMA Spektrumu** | $f = \frac{\sigma^2}{2\pi}\frac{|\theta(e^{i\lambda})|^2}{|\phi(e^{i\lambda})|^2}$ | Model bazlı tahmin |

